In [ ]:
#| default_exp genetic

This derivative-free optimiser minimises a scalar callable `objective(x, y)` using binary DNA, selection, one-point crossover and bit mutation. Fitness is exactly the negative objective: finite negative, positive and tied losses are all supported, without clipping or gradients.

Unlike the experimental implementation's raw floating-point bits, each coordinate uses **24-bit fixed-point encoding**. Every possible chromosome therefore decodes to finite coordinates in a fixed rectangle. `max_offset` bounds **each coordinate** around the supplied starting point throughout the run; it is not a Euclidean-radius limit. Each coordinate has $2^{24}$ levels, spaced by its bound width divided by $2^{24}-1$. The initial population is uniformly sampled in that rectangle and includes the quantised starting point. A zero offset keeps the entire population at the exact starting point.

Each generation retains the best 30% unchanged, selects 40% proportionally to shifted fitness, and fills the remainder with two-way tournaments sampled with replacement. Counts round down, with at least one elite. Selected parents are shuffled and crossed in pairs to produce **two** children per pair; an unpaired parent is retained before mutation. Only non-elites mutate, so the best loss cannot worsen for a deterministic objective.

In [ ]:
#| exporti
from copy import deepcopy
from dataclasses import dataclass
from numbers import Integral, Real

import numpy as np

_BITS_PER_COORDINATE = 24
_MAX_GENE = (1 << _BITS_PER_COORDINATE) - 1
_BIT_SHIFTS = np.arange(_BITS_PER_COORDINATE - 1, -1, -1, dtype=np.uint32)
_BIT_WEIGHTS = np.uint32(1) << _BIT_SHIFTS

In [ ]:
#| export
@dataclass(frozen=True)
class State:
    """A population snapshot: binary DNA, [lower, upper] bounds and private RNG state.

    Fitness is None until the first update, then the negative objective per individual.
    Treat arrays and the RNG dictionary as read-only; updates never modify them.
    """

    generation: np.ndarray
    bounds: np.ndarray
    rng_state: dict
    fitness: np.ndarray | None = None

In [ ]:
#| exporti
def _encode(params, bounds):
    params = np.asarray(params, dtype=np.float64)
    width = bounds[1] - bounds[0]
    fractions = np.divide(params - bounds[0], width,
                          out=np.zeros_like(params), where=width != 0)
    genes = np.rint(np.clip(fractions, 0, 1) * _MAX_GENE).astype(np.uint32)
    return ((genes[..., None] >> _BIT_SHIFTS) & 1).reshape(-1, 48).astype(np.uint8)


def _decode(generation, bounds):
    genes = generation.reshape(-1, 2, _BITS_PER_COORDINATE) @ _BIT_WEIGHTS
    params = bounds[0] + (genes / _MAX_GENE) * (bounds[1] - bounds[0])
    return np.clip(params, bounds[0], bounds[1])


def _real_option(value, name, minimum, maximum=None):
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, Real):
        raise ValueError(f'{name} must be a finite real number')
    try:
        value = float(value)
    except OverflowError as error:
        raise ValueError(f'{name} must be finite') from error
    if not np.isfinite(value) or value < minimum or (maximum is not None and value > maximum):
        limit = f'[{minimum}, {maximum}]' if maximum is not None else f'>= {minimum}'
        raise ValueError(f'{name} must be finite and {limit}')
    return value


def _starting_bounds(params, max_offset):
    try:
        params = np.asarray(params)
    except (TypeError, ValueError) as error:
        raise ValueError('params must contain exactly two finite real numbers') from error
    if params.shape != (2,) or params.dtype.kind not in 'iuf':
        raise ValueError('params must contain exactly two finite real numbers')
    with np.errstate(over='ignore', invalid='ignore'):
        params = params.astype(np.float64)
        bounds = np.array([params - max_offset, params + max_offset])
        width = bounds[1] - bounds[0]
    if not np.all(np.isfinite(params)):
        raise ValueError('params must contain exactly two finite real numbers')
    if (not np.all(np.isfinite(bounds)) or not np.all(np.isfinite(width))
            or (max_offset > 0 and np.any(width <= 0))):
        raise ValueError('max_offset must give finite, representable, nondegenerate bounds')
    return params, bounds

In [ ]:
#| export
def get_params(state):
    """Decode a snapshot into a fresh float64 array of shape (population_size, 2)."""
    return _decode(state.generation, state.bounds)

In [ ]:
#| exporti
def _evaluate(objective, params):
    if not callable(objective):
        raise ValueError('objective must be callable as objective(x, y)')
    fitness = np.empty(len(params), dtype=np.float64)
    for index, (x, y) in enumerate(params):
        loss = np.asarray(objective(x, y))
        if loss.shape != () or loss.dtype.kind not in 'iuf':
            raise ValueError('objective(x, y) must return a finite real scalar')
        value = float(loss)
        if not np.isfinite(value):
            raise ValueError('objective(x, y) must return a finite real scalar')
        fitness[index] = -value
    return fitness


def _proportional_probabilities(fitness):
    # Scale before subtracting: opposite, very large finite values must not overflow.
    scaled = fitness / max(1.0, np.max(np.abs(fitness)))
    weights = scaled - scaled.min()
    total = weights.sum()
    return weights / total if total > 0 else None


def _crossover(parents, rng):
    children = parents[rng.permutation(len(parents))].copy()
    for first in range(0, len(children) - 1, 2):
        locus = rng.integers(1, children.shape[1])
        children[first:first + 2, locus:] = children[first:first + 2, locus:][::-1].copy()
    return children


def _next_generation(generation, fitness, mutation_chance, rng):
    count = len(generation)
    elite_count = max(1, int(0.3 * count))
    proportional_count = int(0.4 * count)
    tournament_count = count - elite_count - proportional_count
    elites = generation[np.argsort(-fitness, kind='stable')[:elite_count]]
    proportional = rng.choice(count, size=proportional_count, replace=True,
                              p=_proportional_probabilities(fitness))
    competitors = rng.integers(count, size=(tournament_count, 2))
    winners = competitors[np.arange(tournament_count), np.argmax(fitness[competitors], axis=1)]
    parents = generation[np.concatenate([proportional, winners])]
    children = _crossover(parents, rng)
    children ^= rng.random(children.shape) < mutation_chance
    return np.concatenate([elites, children])

## Optimiser contract and controls

`genetic_algo(...)` returns `(init, update, get_params)`, compatible with `optimize(...).using(..., derivatives_based=False)`. `init([x, y])` accepts two finite real coordinates, casts them to float64 and returns a `State`. `update(i, objective, state)` returns a new snapshot whose fitness corresponds to its decoded population; `i` is accepted for optimiser compatibility. The objective is evaluated again on the input population, so changing the objective between updates does not reuse stale fitness. Objective exceptions propagate; non-scalar or non-finite returns are rejected.

- `population_size=50`: an integer of at least two; `None` preserves automatic sizing at 200 for two coordinates.
- `max_offset=4`: a finite, nonnegative coordinate bound. Nonzero bounds must remain finite and distinct in float64.
- `mutation_chance=0.01`: the probability of flipping each non-elite bit, between zero and one inclusive.
- `seed=None`: a NumPy random seed. A fixed seed gives reproducible populations and restarts; each state saves its own RNG snapshot, so replaying an update is deterministic and independent of other runs or the global NumPy RNG.

The initial state has `fitness=None`. Updates do not mutate input arrays, previous generations, fitness or RNG dictionaries. `get_params` returns a fresh array, and history snapshots may safely be retained. As with other optimiser state objects, callers should not edit their internal arrays.

In [ ]:
#| export
def genetic_algo(population_size=50, max_offset=4, *, mutation_chance=0.01, seed=None):
    """Return (init, update, get_params) for bounded, binary genetic minimisation.

    Use with derivatives_based=False. max_offset bounds each coordinate around
    the starting point; coordinates use 24-bit fixed-point precision. A fixed
    seed makes initialisation, restarts and independent runs reproducible.
    """
    if population_size is None:
        population_size = 200
    if (isinstance(population_size, (bool, np.bool_))
            or not isinstance(population_size, Integral) or population_size < 2):
        raise ValueError('population_size must be an integer >= 2, or None')
    population_size = int(population_size)
    max_offset = _real_option(max_offset, 'max_offset', 0)
    mutation_chance = _real_option(mutation_chance, 'mutation_chance', 0, 1)
    initial_rng = np.random.default_rng(seed)
    bit_generator_type = type(initial_rng.bit_generator)
    initial_rng_state = deepcopy(initial_rng.bit_generator.state)

    def restore_rng(rng_state):
        rng = np.random.Generator(bit_generator_type(0))
        rng.bit_generator.state = deepcopy(rng_state)
        return rng

    def init(params):
        params, bounds = _starting_bounds(params, max_offset)
        rng = restore_rng(initial_rng_state)
        population = bounds[0] + rng.random((population_size, 2)) * (bounds[1] - bounds[0])
        population[0] = params
        return State(generation=_encode(population, bounds), bounds=bounds,
                     rng_state=deepcopy(rng.bit_generator.state))

    def update(i, objective, state):
        rng = restore_rng(state.rng_state)
        fitness = _evaluate(objective, get_params(state))
        generation = _next_generation(state.generation, fitness, mutation_chance, rng)
        bounds = state.bounds.copy()
        return State(generation=generation, bounds=bounds,
                     fitness=_evaluate(objective, _decode(generation, bounds)),
                     rng_state=deepcopy(rng.bit_generator.state))

    return init, update, get_params

## Seeded optimisation

No JAX-compatible objective or derivative is needed. This NumPy sphere example searches the rectangle `[-1, 7] x [-7, 1]`. Inspect the best individual with `get_params`, rather than treating the entire population as a single point.

In [ ]:
from optimisations.optimizers import optimize

In [ ]:
def sphere(x, y):
    return np.square(x) + np.square(y)

run = (optimize(sphere)
       .using(genetic_algo(population_size=50, max_offset=4, seed=7),
              name='ga', derivatives_based=False)
       .start_from([3.0, -3.0]))
history = run.update(30)
population = get_params(history[-1])
best = population[np.argmax(history[-1].fitness)]
best, sphere(*best)

In [ ]:
#| include: false
losses = [min(sphere(x, y) for x, y in get_params(state)) for state in history]
assert len(history) == 31
assert np.all(np.diff(losses) <= 0)
assert losses[-1] < losses[0] * 0.1
assert history[0].fitness is None
assert all(get_params(state).shape == (50, 2) for state in history)
assert np.all(population >= [ -1, -7]) and np.all(population <= [7, 1])